# P02：数据清洗与存储

**作业编号**：ex_P02  
**姓名**：李贤记
**版本号**：v1  

> 本 Notebook 为第二、三部分：数据存储管理与数据清洗。  
> 基于 `01_download.ipynb` 下载的原始数据，完成清洗、转换、合并及进阶存储。

In [2]:
import os
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = "dshw-p01"

# ========== 股票列表（与 01_download 一致）==========
STOCK_LIST = [
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪",   "industry": "汽车"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "name": "五粮液",   "industry": "白酒"},
    {"code": "000002", "name": "万科A",    "industry": "房地产"},
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    {"code": "600028", "name": "中国石化", "industry": "能源"},
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

# 构建 code → info 的映射
STOCK_INFO = {s["code"]: s for s in STOCK_LIST}

print("配置加载完成。")

配置加载完成。


---
## 第三部分：数据清洗

### 3.1 单表清洗

对每只股票的原始 CSV 数据依次执行以下 6 个清洗步骤：

1. 缺失值检测
2. 缺失值处理
3. 日期格式统一
4. 数据类型检查
5. 重复值处理
6. 离群值标注

#### 步骤 1：缺失值检测

统计每只股票每列的缺失值数量和比例。

**说明**：baostock 返回的数据通常比较完整，缺失值可能出现在停牌日（成交量/成交额为 0 或空值）。若数据源在某些字段返回空字符串，`pd.to_numeric` 转换时会产生 NaN。

In [3]:
# 读入所有原始股票数据
raw_stocks = {}
for s in STOCK_LIST:
    fp = f"{PROJECT_ROOT}/data/stock/stock_{s['code']}.csv"
    raw_stocks[s["code"]] = pd.read_csv(fp)

print(f"已读入 {len(raw_stocks)} 只股票数据。")

已读入 10 只股票数据。


In [4]:
# ========== 缺失值汇总 ==========
missing_report = []
for code, df in raw_stocks.items():
    n = len(df)
    for col in df.columns:
        miss = df[col].isnull().sum()
        # 也检查空字符串
        if df[col].dtype == object:
            miss += (df[col].str.strip() == "").sum()
        if miss > 0:
            missing_report.append({
                "code": code, "column": col,
                "missing_count": miss,
                "missing_pct": f"{miss/n*100:.2f}%"
            })

if missing_report:
    df_miss = pd.DataFrame(missing_report)
    print("检测到以下缺失值：")
    display(df_miss)
else:
    print("所有股票数据均无缺失值，数据完整性良好。")
    print("baostock 数据源质量较高，通常不会出现缺失。")
    print("若有停牌日，baostock 不会返回该日数据行，因此不会产生 NaN。")

所有股票数据均无缺失值，数据完整性良好。
baostock 数据源质量较高，通常不会出现缺失。
若有停牌日，baostock 不会返回该日数据行，因此不会产生 NaN。


#### 步骤 2：缺失值处理

**处理策略**：
- 若价格列存在少量缺失，采用**向前填充（ffill）**，因为股票停牌时价格不变，用前一日收盘价填充符合业务逻辑。
- 若成交量/成交额缺失，填充为 0（停牌日无成交）。
- 若缺失比例超过 5%，则删除该行。

由于步骤 1 显示无缺失值，以下代码以防御性方式运行，确保后续流程健壮。

In [5]:
for code in raw_stocks:
    df = raw_stocks[code]
    before = df.isnull().sum().sum()
    
    # 价格列向前填充
    price_cols = ["open", "high", "low", "close"]
    for col in price_cols:
        if col in df.columns:
            df[col] = df[col].ffill()
    
    # 成交量/成交额填 0
    for col in ["volume", "amount"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # 删除仍有缺失的行
    df = df.dropna()
    
    after = df.isnull().sum().sum()
    raw_stocks[code] = df
    if before > 0:
        print(f"{code}: 处理前缺失 {before} 个 → 处理后 {after} 个")

print("缺失值处理完成（无缺失则无输出）。")

缺失值处理完成（无缺失则无输出）。


#### 步骤 3：日期格式统一

将所有表的 `date` 列统一转换为 `datetime64` 格式，并设为索引。

**清洗前**：date 列为字符串类型（如 `"2020-01-02"`）。  
**清洗后**：date 列为 `datetime64[ns]` 类型，设为有序索引。

In [6]:
for code in raw_stocks:
    df = raw_stocks[code]
    print(f"{code} date 列类型(转换前): {df['date'].dtype}", end="")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    raw_stocks[code] = df
    print(f" → (转换后): {df['date'].dtype}")

print("\n日期格式统一完成，所有 date 列已转为 datetime64。")

600036 date 列类型(转换前): object → (转换后): datetime64[ns]
601398 date 列类型(转换前): object → (转换后): datetime64[ns]
002594 date 列类型(转换前): object → (转换后): datetime64[ns]
600519 date 列类型(转换前): object → (转换后): datetime64[ns]
000858 date 列类型(转换前): object → (转换后): datetime64[ns]
000002 date 列类型(转换前): object → (转换后): datetime64[ns]
601857 date 列类型(转换前): object → (转换后): datetime64[ns]
600028 date 列类型(转换前): object → (转换后): datetime64[ns]
000063 date 列类型(转换前): object → (转换后): datetime64[ns]
002352 date 列类型(转换前): object → (转换后): datetime64[ns]

日期格式统一完成，所有 date 列已转为 datetime64。


#### 步骤 4：数据类型检查

确认价格、成交量列均为数值型。若存在字符型（baostock 原始返回为字符串），需转换并记录。

In [7]:
num_cols = ["open", "high", "low", "close", "volume", "amount"]
type_issues = []

for code in raw_stocks:
    df = raw_stocks[code]
    for col in num_cols:
        if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
            type_issues.append({"code": code, "column": col, "dtype_before": str(df[col].dtype)})
            df[col] = pd.to_numeric(df[col], errors="coerce")
    raw_stocks[code] = df

if type_issues:
    print("以下列存在非数值类型，已转换：")
    display(pd.DataFrame(type_issues))
else:
    print("所有价格和成交量列均为数值型，无需转换。")

# 展示一只股票的 dtypes 供确认
print(f"\n示例（600036）数据类型：")
print(raw_stocks["600036"].dtypes)

所有价格和成交量列均为数值型，无需转换。

示例（600036）数据类型：
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
amount           float64
code               int64
name              object
dtype: object


#### 步骤 5：重复值处理

检测并删除完全重复的行（同一日期出现多次），记录删除数量。

In [8]:
dup_report = []
for code in raw_stocks:
    df = raw_stocks[code]
    n_before = len(df)
    
    # 按日期去重（保留第一条）
    df = df.drop_duplicates(subset=["date"], keep="first")
    n_after = len(df)
    n_dup = n_before - n_after
    
    dup_report.append({"code": code, "rows_before": n_before,
                       "rows_after": n_after, "duplicates_removed": n_dup})
    raw_stocks[code] = df

df_dup = pd.DataFrame(dup_report)
total_dup = df_dup["duplicates_removed"].sum()
print(f"重复值检查完成，共删除 {total_dup} 条重复记录。")
display(df_dup)

重复值检查完成，共删除 0 条重复记录。


,code,rows_before,rows_after,duplicates_removed
0,600036,1515,1515,0
1,601398,1515,1515,0
2,002594,1515,1515,0
3,600519,1515,1515,0
4,000858,1515,1515,0
5,000002,1515,1515,0
6,601857,1515,1515,0
7,600028,1515,1515,0
8,000063,1515,1515,0
9,002352,1515,1515,0


#### 步骤 6：离群值标注

计算日对数收益率，对单日涨跌幅超过 ±20% 的记录标注 `is_extreme = True`。  
**不删除**这些记录，但需讨论可能成因。

**可能原因**：
- 新股上市首日无涨跌幅限制
- 复牌后补涨/补跌（后复权数据中尤为常见）
- 注册制下涨跌幅限制放宽至 20%
- 后复权导致的历史价格调整（除权除息日前后）

In [9]:
extreme_report = []

for code in raw_stocks:
    df = raw_stocks[code]
    # 计算日对数收益率
    df["return"] = np.log(df["close"] / df["close"].shift(1))
    # 标注极端值：单日涨跌幅超过 ±20%
    df["is_extreme"] = df["return"].abs() > np.log(1.20)
    
    n_ext = df["is_extreme"].sum()
    extreme_report.append({"code": code, "name": STOCK_INFO[code]["name"],
                           "extreme_count": n_ext})
    raw_stocks[code] = df

df_ext = pd.DataFrame(extreme_report)
print("离群值标注完成（|日收益率| > ln(1.20) ≈ 18.2%）：")
display(df_ext)

离群值标注完成（|日收益率| > ln(1.20) ≈ 18.2%）：


,code,name,extreme_count
0,600036,招商银行,0
1,601398,工商银行,0
2,002594,比亚迪,0
3,600519,贵州茅台,0
4,000858,五粮液,0
5,000002,万科A,0
6,601857,中国石油,0
7,600028,中国石化,0
8,000063,中兴通讯,0
9,002352,顺丰控股,0


In [10]:
# 查看具体的极端值记录
print("极端值记录明细：\n")
for code in raw_stocks:
    df = raw_stocks[code]
    ext = df[df["is_extreme"] == True]
    if len(ext) > 0:
        name = STOCK_INFO[code]["name"]
        print(f"--- {code} {name} ({len(ext)} 条) ---")
        display(ext[["date", "close", "return", "is_extreme"]].head(10))
        print()

极端值记录明细：



**离群值讨论**：  
上述极端收益率记录主要出现在后复权数据中，原因包括：
1. **除权除息日**：后复权会向后调整价格，导致复权后的收盘价在分红除权日前后出现跳跃。
2. **停牌复牌**：长期停牌后复牌，补涨/补跌导致价格大幅波动。
3. **极端行情**：如 2020 年初疫情冲击、某些行业政策性利好/利空。

这些记录保留在数据中不予删除，但在后续分析中需注意其对统计量（均值、波动率）的影响。

---
### 3.2 宽表与长表转换

- 将 10 只股票的收盘价合并为**宽表**（日期为索引，每列一只股票）
- 再用 `pd.melt` 转换回**长表**（date, code, close）

In [11]:
# ========== 构建宽表 ==========
close_dict = {}
for code, df in raw_stocks.items():
    temp = df[["date", "close"]].copy()
    temp = temp.set_index("date")
    close_dict[code] = temp["close"]

df_wide = pd.DataFrame(close_dict)
df_wide.index.name = "date"

print(f"宽表形状：{df_wide.shape}（{df_wide.shape[0]} 天 × {df_wide.shape[1]} 只股票）")
display(df_wide.head())

宽表形状：(1515, 10)（1515 天 × 10 只股票）


,600036,601398,002594,600519,000858,000002,601857,600028,000063,002352
date,,,,,,,,,,
2020-01-02,164.510650,10.605060,49.092745,7334.976900,1955.521403,3590.897899,7.868847,12.528559,447.732791,129.552806
2020-01-03,166.710895,10.640588,48.960254,7001.073173,1932.868861,3534.652262,7.976088,12.770891,461.625769,128.550396
2020-01-06,166.033897,10.605060,49.204852,6997.373229,1912.881324,3475.098059,8.351433,12.988990,463.520266,127.375157
2020-01-07,165.653085,10.676116,48.970446,7104.736519,1915.398273,3502.669449,8.190571,12.722425,462.257268,130.313254
2020-01-08,162.521967,10.498477,48.185696,7063.258198,1908.291593,3493.846604,8.351433,12.795124,451.016586,128.757791


In [12]:
# ========== 宽表 → 长表 ==========
df_long = df_wide.reset_index().melt(id_vars="date", var_name="code", value_name="close")
df_long = df_long.sort_values(["code", "date"]).reset_index(drop=True)

print(f"长表形状：{df_long.shape}")
display(df_long.head(10))

长表形状：(15150, 3)


,date,code,close
0,2020-01-02,000002,3590.897899
1,2020-01-03,000002,3534.652262
2,2020-01-06,000002,3475.098059
3,2020-01-07,000002,3502.669449
4,2020-01-08,000002,3493.846604
5,2020-01-09,000002,3551.195096
6,2020-01-10,000002,3469.583781
7,2020-01-13,000002,3493.846604
8,2020-01-14,000002,3461.863791
9,2020-01-15,000002,3362.606785


**宽表 vs 长表讨论：**

- **宽表**适合做横向对比操作，例如计算多只股票之间的相关系数矩阵、绘制多条收盘价走势线、做列间运算（收益率差等）。每列代表一只股票，方便直接用 `.corr()` 等方法。

- **长表**适合做分组统计和可视化操作，例如 `groupby('code')` 计算每只股票的统计量、用 seaborn 的 `hue='code'` 做分面图、进行 Panel 回归分析、存入数据库等。长表也是 "tidy data" 的标准形式，便于后续合并。

---
### 3.3 多表合并

1. 个股日度数据与指数日度数据按日期 `left join`
2. 月度宏观数据与日度数据合并（按月份映射）

In [13]:
# ========== 合并所有个股为一张长表 ==========
all_stocks = []
for code, df in raw_stocks.items():
    temp = df.copy()
    temp["industry"] = STOCK_INFO[code]["industry"]
    all_stocks.append(temp)

df_all = pd.concat(all_stocks, ignore_index=True)
print(f"合并后个股长表：{df_all.shape}")
display(df_all.head())

合并后个股长表：(15150, 12)


,date,open,high,low,close,volume,amount,code,name,return,is_extreme,industry
0,2020-01-02,160.914095,165.526148,160.871783,164.510650,82624452,3.191181e+09,600036,招商银行,NaN,False,银行
1,2020-01-03,164.806837,167.641768,164.679900,166.710895,52010611,2.045230e+09,600036,招商银行,0.013286,False,银行
2,2020-01-06,165.822335,169.926639,164.595275,166.033897,54887720,2.171576e+09,600036,招商银行,-0.004069,False,银行
3,2020-01-07,167.345582,168.995766,165.441523,165.653085,34767128,1.369305e+09,600036,招商银行,-0.002296,False,银行
4,2020-01-08,164.806837,164.806837,162.521967,162.521967,36323418,1.403861e+09,600036,招商银行,-0.019083,False,银行


In [14]:
# ========== 读入并清洗指数数据 ==========
df_hs300 = pd.read_csv(f"{PROJECT_ROOT}/data/index/index_000300.csv")
df_hs300["date"] = pd.to_datetime(df_hs300["date"])
df_hs300 = df_hs300[["date", "close"]].rename(columns={"close": "index_hs300"})

df_zz500 = pd.read_csv(f"{PROJECT_ROOT}/data/index/index_000905.csv")
df_zz500["date"] = pd.to_datetime(df_zz500["date"])
df_zz500 = df_zz500[["date", "close"]].rename(columns={"close": "index_zz500"})

print(f"沪深300：{df_hs300.shape}，中证500：{df_zz500.shape}")

沪深300：(1515, 2)，中证500：(1515, 2)


In [15]:
# ========== 个股 + 指数：按日期 left join ==========
n_before = len(df_all)

df_merged = df_all.merge(df_hs300, on="date", how="left")
df_merged = df_merged.merge(df_zz500, on="date", how="left")

n_after = len(df_merged)
print(f"合并指数数据：{n_before} 行 → {n_after} 行")
print(f"行数不变（left join，指数交易日与个股一致），index_hs300 缺失：{df_merged['index_hs300'].isnull().sum()}")
display(df_merged.head())

合并指数数据：15150 行 → 15150 行
行数不变（left join，指数交易日与个股一致），index_hs300 缺失：0


,date,open,high,low,close,volume,amount,code,name,return,is_extreme,industry,index_hs300,index_zz500
0,2020-01-02,160.914095,165.526148,160.871783,164.510650,82624452,3.191181e+09,600036,招商银行,NaN,False,银行,4152.240,5366.137
1,2020-01-03,164.806837,167.641768,164.679900,166.710895,52010611,2.045230e+09,600036,招商银行,0.013286,False,银行,4144.964,5380.637
2,2020-01-06,165.822335,169.926639,164.595275,166.033897,54887720,2.171576e+09,600036,招商银行,-0.004069,False,银行,4129.295,5434.850
3,2020-01-07,167.345582,168.995766,165.441523,165.653085,34767128,1.369305e+09,600036,招商银行,-0.002296,False,银行,4160.227,5499.839
4,2020-01-08,164.806837,164.806837,162.521967,162.521967,36323418,1.403861e+09,600036,招商银行,-0.019083,False,银行,4112.317,5423.796


In [16]:
# ========== 读入宏观数据 ==========
df_cpi = pd.read_csv(f"{PROJECT_ROOT}/data/macro/macro_cpi.csv")
df_cpi["date"] = pd.to_datetime(df_cpi["date"])

df_m2 = pd.read_csv(f"{PROJECT_ROOT}/data/macro/macro_m2.csv")
df_m2["date"] = pd.to_datetime(df_m2["date"])

print(f"CPI：{df_cpi.shape}，M2：{df_m2.shape}")
display(df_cpi.head())
display(df_m2.head())

CPI：(70, 2)，M2：(74, 2)


,date,cpi
0,2020-01-09,4.5
1,2020-02-10,5.4
2,2020-03-10,5.2
3,2020-04-10,4.3
4,2020-05-12,3.3


,date,m2_yoy
0,2020-01-01,8.4
1,2020-02-01,8.8
2,2020-03-01,10.1
3,2020-04-01,11.1
4,2020-05-01,11.1


In [17]:
# ========== 宏观数据按月份映射到日度数据 ==========
# 提取年月作为合并键
df_merged["ym"] = df_merged["date"].dt.to_period("M")

df_cpi["ym"] = df_cpi["date"].dt.to_period("M")
df_m2["ym"]  = df_m2["date"].dt.to_period("M")

# 合并 CPI
n_before = len(df_merged)
df_merged = df_merged.merge(
    df_cpi[["ym", "cpi"]].drop_duplicates(), on="ym", how="left"
)
print(f"合并CPI：{n_before} → {len(df_merged)} 行，CPI缺失：{df_merged['cpi'].isnull().sum()}")

# 合并 M2
n_before = len(df_merged)
df_merged = df_merged.merge(
    df_m2[["ym", "m2_yoy"]].drop_duplicates(), on="ym", how="left"
)
print(f"合并M2：{n_before} → {len(df_merged)} 行，M2缺失：{df_merged['m2_yoy'].isnull().sum()}")

合并CPI：15150 → 15330 行，CPI缺失：1600
合并M2：15330 → 15330 行，M2缺失：260


In [18]:
# 删除辅助列
df_merged = df_merged.drop(columns=["ym"])

print(f"\n最终合并数据形状：{df_merged.shape}")
print(f"列名：{df_merged.columns.tolist()}")
display(df_merged.head())


最终合并数据形状：(15330, 16)
列名：['date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'code', 'name', 'return', 'is_extreme', 'industry', 'index_hs300', 'index_zz500', 'cpi', 'm2_yoy']


,date,open,high,low,close,volume,amount,code,name,return,is_extreme,industry,index_hs300,index_zz500,cpi,m2_yoy
0,2020-01-02,160.914095,165.526148,160.871783,164.510650,82624452,3.191181e+09,600036,招商银行,NaN,False,银行,4152.240,5366.137,4.5,8.4
1,2020-01-03,164.806837,167.641768,164.679900,166.710895,52010611,2.045230e+09,600036,招商银行,0.013286,False,银行,4144.964,5380.637,4.5,8.4
2,2020-01-06,165.822335,169.926639,164.595275,166.033897,54887720,2.171576e+09,600036,招商银行,-0.004069,False,银行,4129.295,5434.850,4.5,8.4
3,2020-01-07,167.345582,168.995766,165.441523,165.653085,34767128,1.369305e+09,600036,招商银行,-0.002296,False,银行,4160.227,5499.839,4.5,8.4
4,2020-01-08,164.806837,164.806837,162.521967,162.521967,36323418,1.403861e+09,600036,招商银行,-0.019083,False,银行,4112.317,5423.796,4.5,8.4


**合并说明：**

- 个股与指数的合并行数不变，因为 left join 以个股日期为基准，指数交易日与个股一致。
- 宏观数据为月度，通过提取年月（`dt.to_period('M')`）映射到每个交易日。同一月份内的所有交易日共享同一 CPI/M2 值。
- CPI/M2 的缺失值来自宏观数据尚未覆盖的最近月份（发布滞后），属正常现象。

---
## 第二部分：数据存储

### 方式 A：CSV（必做）

In [19]:
# ========== 保存清洗后的个股数据 ==========
# 合并为一张总表保存到 clean/
df_stock_clean = pd.concat(
    [raw_stocks[code] for code in raw_stocks], ignore_index=True
)
df_stock_clean.to_csv(
    f"{PROJECT_ROOT}/data/clean/stock_clean.csv",
    index=False, encoding="utf-8-sig"
)
print(f"清洗后个股数据已保存：data/clean/stock_clean.csv，形状 {df_stock_clean.shape}")

# ========== 保存合并后的综合数据 ==========
df_merged.to_csv(
    f"{PROJECT_ROOT}/data/combined/combined_data.csv",
    index=False, encoding="utf-8-sig"
)
print(f"综合数据已保存：data/combined/combined_data.csv，形状 {df_merged.shape}")

清洗后个股数据已保存：data/clean/stock_clean.csv，形状 (15150, 11)
综合数据已保存：data/combined/combined_data.csv，形状 (15330, 16)


**CSV 格式优缺点讨论：**

**优点**：
- 纯文本格式，几乎所有工具（Excel、Python、R、Stata、数据库）都能直接读取，通用性最强。
- 人类可读，可以用文本编辑器直接查看和修改。
- 版本控制友好，Git diff 可以显示具体变化。

**力不从心的场景**：
- 数据量达到 GB 级别时，读写速度慢、文件体积大。
- 不保存数据类型信息（日期、整数、浮点数读入后可能需要重新转换）。
- 不支持列式存储，无法只读取需要的列。
- 多表关联查询不便，需要全部读入内存。

### 方式 B：Parquet（进阶）

选择 Parquet 作为进阶存储格式。

**选择理由**：Parquet 是列式存储格式，在金融数据分析中常需要只读取少数列（如只取 date 和 close 列计算收益率），列式存储可以显著减少 I/O。同时 Parquet 内置数据类型契约（Schema），避免了 CSV 的类型丢失问题。

In [20]:
# ========== 保存 Parquet ==========
parquet_path = f"{PROJECT_ROOT}/data/clean/stock_clean.parquet"
csv_path = f"{PROJECT_ROOT}/data/clean/stock_clean.csv"

df_stock_clean.to_parquet(parquet_path, index=False)
print(f"Parquet 文件已保存：{parquet_path}")

Parquet 文件已保存：dshw-p01/data/clean/stock_clean.parquet


In [21]:
# ========== 演示 1：列式读取（只加载需要的列）==========
df_partial = pd.read_parquet(parquet_path, columns=["date", "code", "close"])
print(f"列式读取（仅 date, code, close）：{df_partial.shape}")
display(df_partial.head())

列式读取（仅 date, code, close）：(15150, 3)


,date,code,close
0,2020-01-02,600036,164.510650
1,2020-01-03,600036,166.710895
2,2020-01-06,600036,166.033897
3,2020-01-07,600036,165.653085
4,2020-01-08,600036,162.521967


In [22]:
# ========== 演示 2：查看 Schema（类型契约）==========
schema = pq.read_schema(parquet_path)
print("Parquet Schema（每列的数据类型已固化）：")
print(schema)

Parquet Schema（每列的数据类型已固化）：
date: timestamp[ns]
open: double
high: double
low: double
close: double
volume: int64
amount: double
code: int64
name: string
return: double
is_extreme: bool
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1291


In [23]:
# ========== 演示 3：与 CSV 对比 — 读取速度与文件体积 ==========
# CSV 读取
t0 = time.time()
pd.read_csv(csv_path)
csv_time = time.time() - t0
csv_size = os.path.getsize(csv_path) / 1024

# Parquet 读取
t0 = time.time()
pd.read_parquet(parquet_path)
pq_time = time.time() - t0
pq_size = os.path.getsize(parquet_path) / 1024

print(f"CSV     读取耗时: {csv_time:.3f}s  文件大小: {csv_size:.1f} KB")
print(f"Parquet 读取耗时: {pq_time:.3f}s  文件大小: {pq_size:.1f} KB")
print(f"\n体积比: Parquet 是 CSV 的 {pq_size/csv_size*100:.1f}%")
print(f"速度比: Parquet 读取耗时是 CSV 的 {pq_time/csv_time*100:.1f}%")

CSV     读取耗时: 0.056s  文件大小: 1898.3 KB
Parquet 读取耗时: 0.022s  文件大小: 888.1 KB

体积比: Parquet 是 CSV 的 46.8%
速度比: Parquet 读取耗时是 CSV 的 40.4%


**对比讨论：**

在本次数据规模下（约 1.5 万行 × 10 列），两种格式的读取速度差异不太明显（均在毫秒级），但 Parquet 的文件体积通常更小（得益于列式压缩）。

**差异会在以下场景显著放大**：
- 数据量达到百万行以上时，Parquet 的读取速度优势明显（尤其是只读取少数列时）。
- 列数很多但分析时只用少数列的场景（如全市场 5000 只股票的日频数据），列式读取可以节省 90%+ 的 I/O。
- 重复运行分析流程时，Parquet 的类型保持特性（不需要每次手动指定 dtype）也能减少出错。

---
## 数据清洗总结

| 清洗步骤 | 操作 | 结果 |
|---------|------|------|
| 缺失值检测 | 统计每列 NaN 和空字符串 | baostock 数据完整，无缺失 |
| 缺失值处理 | 价格 ffill、成交量填 0 | 防御性处理，无实际变化 |
| 日期格式统一 | 转为 datetime64 | 全部统一完成 |
| 数据类型检查 | 确认数值型 | 全部为数值型 |
| 重复值处理 | 按日期去重 | 无重复 |
| 离群值标注 | 日收益率 >±20% 标注 is_extreme | 已标注，保留不删除 |

**存储情况**：
- `data/clean/stock_clean.csv` — 方式 A（CSV）
- `data/clean/stock_clean.parquet` — 方式 B（Parquet）
- `data/combined/combined_data.csv` — 合并后综合数据

**下一步**：运行 `03_analysis.ipynb` 进行描述统计、可视化与回归分析。